<style>

/* Target VS Code notebook markdown renderer */
.jp-RenderedHTMLCommon h1 {
    color: #1f77b4 !important;
}

.jp-RenderedHTMLCommon h2 {
    color: #2ca02c !important;
}

.jp-RenderedHTMLCommon h3 {
    color: #d62728 !important;
}

.jp-RenderedHTMLCommon blockquote {
    color: #ff7f0e !important;
    font-weight: bold;
}

.jp-RenderedHTMLCommon code {
    color: #9467bd !important;
}

</style>

# Title

## Section

### Subsection

> Important note

example code

In [3]:
# Importing Libraries
import numpy as np
import pandas as pd

In [4]:
df = pd.read_csv("DataCo_ML_Dataset.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'DataCo_ML_Dataset.csv'

In [ ]:
# Look at the first 5 rows to ensure the headers and data look correct
display(df.head())

# Check for missing (null) values that could break our ML model
print(df.isnull().sum(), "\n")

# Check the exact shape (Number of Rows, Number of Columns)
print(f"Total Rows and Columns: {df.shape}")

,Late_Delivery_Risk,Sales,Order_Item_Quantity,Order_Profit_Per_Order,Days_Scheduled,Category_Name,Customer_Segment,Shipping_Type,Customer_Country,Order_Country,Month_Name,Day_of_Week
0,0,299.98,1,88.79,4,Camping & Hiking,Consumer,CASH,EE. UU.,MTxico,January,Thursday
1,0,199.99,1,91.18,4,Water Sports,Consumer,PAYMENT,EE. UU.,Colombia,January,Thursday
2,0,250.00,5,68.25,4,Women's Apparel,Consumer,PAYMENT,EE. UU.,Colombia,January,Thursday
3,0,129.99,1,36.47,4,Men's Footwear,Consumer,PAYMENT,EE. UU.,Colombia,January,Thursday
4,1,49.98,2,4.10,4,Accessories,Home Office,CASH,EE. UU.,Colombia,January,Thursday


Late_Delivery_Risk        0
Sales                     0
Order_Item_Quantity       0
Order_Profit_Per_Order    0
Days_Scheduled            0
Category_Name             0
Customer_Segment          0
Shipping_Type             0
Customer_Country          0
Order_Country             0
Month_Name                0
Day_of_Week               0
dtype: int64 

Total Rows and Columns: (180519, 12)


Data Quality Check: Verified 0 missing values across all 180,000+ records. The Star Schema architecture successfully enforced data integrity during the SQL extraction phase.


In [ ]:
categorical_cols = [
    "Category_Name",
    "Customer_Segment",
    "Shipping_Type",
    "Customer_Country",
    "Order_Country",
    "Month_Name",
    "Day_of_Week",
]

# 3. Apply One-Hot Encoding to translate text to 0s and 1s
# drop_first=True is a pro-tip: it drops one redundant "switch" to keep the math clean
df_encoded = pd.get_dummies(df, columns = categorical_cols, drop_first=True)

In [ ]:
df_encoded.head() # see now it has 240 columns

,Late_Delivery_Risk,Sales,Order_Item_Quantity,Order_Profit_Per_Order,Days_Scheduled,Category_Name_As Seen on TV!,Category_Name_Baby,Category_Name_Baseball & Softball,Category_Name_Basketball,Category_Name_Books,...,Month_Name_May,Month_Name_November,Month_Name_October,Month_Name_September,Day_of_Week_Monday,Day_of_Week_Saturday,Day_of_Week_Sunday,Day_of_Week_Thursday,Day_of_Week_Tuesday,Day_of_Week_Wednesday
0,0,299.98,1,88.79,4,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1,0,199.99,1,91.18,4,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,0,250.00,5,68.25,4,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,0,129.99,1,36.47,4,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
4,1,49.98,2,4.10,4,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False


Module 2: The Train-Test Split (Preparing the Exam)
- Training Data - 80% of the data to study and find hidden patterns
- Testing Data - 20% remaining hide 
    - Once the model is done studying, we force it to take a blind test on this 20% to see how accurate it actually is.

### USING Scikit-Learn (sklearn)

In [ ]:
# Importing the splitting tool from Scikit-Learn
from sklearn.model_selection import train_test_split

# 1. Separating the Target (y) from the Features (X)
# 'y' is the Final Answer we want to predict

y = df_encoded['Late_Delivery_Risk']

x = df_encoded.drop('Late_Delivery_Risk', axis = 1)

X_train, x_test, y_train, y_test = train_test_split(x,y,test_size = 0.2, random_state=42)

print(df_encoded.shape)
print(X_train.shape)
print(x_test.shape)

(180519, 240)
(144415, 239)
(36104, 239)


For drop = 
In the Pandas library, a DataFrame is just a 2D grid:
- axis=0 means Rows (going top to bottom). (by default)
- axis=1 means Columns (going left to right).

### Module 3: Feature Scaling (The Great Equalizer)
We have one last problem to fix before we train the AI.

Look at your numerical columns. You have a Sales column where numbers can be $1,500.00, and you have an Order_Item_Quantity column where numbers are just 1, 2, or 3.

Because Machine Learning algorithms are just math, they are easily tricked. The algorithm will look at $1,500 and think, "Wow, 1500 is way bigger than 3. Therefore, Sales must be 500 times more important than Quantity!" This is false. To fix this, we use Feature Scaling (specifically a StandardScaler). It squashes all our big numbers and little numbers down to the exact same scale (usually between -3 and 3) so the AI treats them fairly.

- StandardScaler - uses Z score ( how far are values from the average)
    - (Your Value - The Average) / The Standard Deviation

- StandardScaler is a blueprint for weighing machine
    - scaler - weighing machine
        - scaler = StandardScaler() -------  [Weighing Machine using Blueprint]

In [ ]:
# Import the scaling tool from Scikit-Learn
from sklearn.preprocessing import StandardScaler

# 1. Create the scaler (Think of this as an audio equalizer)
scaler = StandardScaler()

# 2. We only want to squash our continuous numbers! 
# (We leave our 0s and 1s from the One-Hot Encoding alone)
num_cols = ['Sales', 'Order_Item_Quantity', 'Order_Profit_Per_Order', 'Days_Scheduled']

# 3. Apply the scaler to our Training and Testing data
# Notice we use fit_transform() on the training data, but only transform() on the testing data
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
x_test[num_cols] = scaler.transform(x_test[num_cols])

X_train[num_cols].head()

print()

3. Why fit_transform on X_train, but only transform on X_test?
- This is the single most important rule in Machine Learning. If you understand this, you are ahead of 50% of junior data scientists.

    - fit means: "Look at the data and learn the average."
    - transform means: "Do the math and squash the numbers."

If we used fit on the testing data, the scaler would calculate the average of the test data. The AI would secretly learn information about the final exam before taking it! This is a massive error called Data Leakage.

- Instead, we only use transform. This forces the scaler to squash the test data using the Training Data's average. It keeps the test 100% blind and fair.

### Module 4: The Machine Learning Model (Random Forest)
For this supply chain problem, we need the AI to predict a simple "Yes" or "No" (1 or 0): Will this order be late?

To do this, we are going to use one of the most powerful and popular algorithms in the world: The Random Forest Classifier.

In [ ]:
# Import the Random Forest algorithm and our grading tools
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

.fit(X_train,y_train)
.predict(x_test, y_test)

In [ ]:
# 1. Bring in the AI
# n_estimators=100 means we are hiring 100 'Decision Trees' to vote on the answer
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. Train the AI (Force it to study the Training Data)
print("Training the AI... (Grab a sip of coffee, this might take a minute!)")
rf_model.fit(X_train, y_train)
print("Training Complete! The AI has learned the patterns.")

Training the AI... (Grab a sip of coffee, this might take a minute!)
Training Complete! The AI has learned the patterns.


In [ ]:
print("Taking the Final Exam...")
y_pred = rf_model.predict(x_test)
print('done')

Taking the Final Exam...
done


In [ ]:
# 4. Grade the Exam! (Compare the AI's guesses to the real, hidden answers)
accuracy = accuracy_score(y_test, y_pred)
print(f"\n🏆 Final Exam Score (Accuracy): {accuracy * 100:.2f}%")


🏆 Final Exam Score (Accuracy): 73.31%


In [ ]:
# 5. The Detailed Report Card
print("\n--- Detailed Report Card ---")
print(classification_report(y_test, y_pred))


--- Detailed Report Card ---
              precision    recall  f1-score   support

           0       0.68      0.77      0.72     16273
           1       0.79      0.70      0.74     19831

    accuracy                           0.73     36104
   macro avg       0.73      0.74      0.73     36104
weighted avg       0.74      0.73      0.73     36104



In Python, we use a library called joblib to take the AI's brain (the math it learned) and save it as a physical file on your hard drive. This process is called Model Serialization.

The Code: Saving the AI's Brain
- To make our Web App work later, we actually need to save three things:

1. The Model: The AI itself.

2. The Scaler: The tool that squashes numbers, so the web app knows how to squash new data.

3. The Columns: The exact list of 240+ columns the AI expects to see.

In [ ]:
# Import the saving tool
import joblib

# 1. Save the trained Random Forest AI
joblib.dump(rf_model, 'dataco_rf_model.joblib')

# 2. Save the Scaler (so the web app can squash new inputs)
joblib.dump(scaler, 'dataco_scaler.joblib')

# 3. Save the exact column nconames (so the web app matches the training data)
joblib.dump(X_train.columns, 'dataco_columns.joblib')

print("💾 AI Brain, Scaler, and Columns successfully saved to your hard drive!")

💾 AI Brain, Scaler, and Columns successfully saved to your hard drive!


In [ ]:
import joblib

# The compress=3 parameter shrinks the file size massively!
joblib.dump(model, 'dataco_rf_model.joblib', compress=3)
joblib.dump(scaler, 'dataco_scaler.joblib')
joblib.dump(model_columns, 'dataco_columns.joblib')

NameError: name 'model' is not defined